In [1]:
import os
import requests
from datetime import datetime, date
import time
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# ---------------------------
# ENV
# ---------------------------
load_dotenv()

POSTGRES_DB = os.getenv("POSTGRES_DB")
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
POLYGON_API_KEY = os.getenv("POLYGON_API_KEY")

if not all([POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD, POLYGON_API_KEY]):
    raise EnvironmentError("Faltan variables de entorno")

# ---------------------------
# Conexión PostgreSQL
# ---------------------------
engine = create_engine(
    f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)

# ---------------------------
# Configuración filtros opciones
# ---------------------------
MIN_DTE = 25
MAX_DTE = 45
MIN_DELTA = -0.35
MAX_DELTA = -0.25
CONTRACT_TYPE = "put"

# ---------------------------
# 1️⃣ Obtener tickers multifactor top 20%
# ---------------------------
query_tickers = """
SELECT DISTINCT ticker
FROM volatilidad.salud_ttm
"""
df_tickers = pd.read_sql(query_tickers, engine)
tickers_validos = df_tickers["ticker"].tolist()

print(f"Procesando {len(tickers_validos)} tickers multifactor top20...")

# ---------------------------
# 2️⃣ Consulta Polygon
# ---------------------------
def buscar_opciones_por_ticker(ticker_raiz):
    hoy = datetime.now()
    url = f"https://api.polygon.io/v3/snapshot/options/{ticker_raiz}?limit=250&apiKey={POLYGON_API_KEY}"

    resultados = []
    buscando = True

    while url and buscando:
        try:
            r = requests.get(url)
            if r.status_code != 200:
                print(f"Error Polygon {ticker_raiz}: {r.status_code}")
                break

            data = r.json()
            contracts = data.get("results", [])

            for c in contracts:
                details = c.get("details", {})
                greeks = c.get("greeks", {})
                day = c.get("day", {})

                if details.get("contract_type") != CONTRACT_TYPE:
                    continue

                expiry = details.get("expiration_date")
                if not expiry:
                    continue

                fecha_vto = datetime.strptime(expiry, "%Y-%m-%d")
                dte = (fecha_vto - hoy).days

                if dte > MAX_DTE:
                    buscando = False
                    break

                delta = greeks.get("delta")
                if delta is None or not (MIN_DELTA <= delta <= MAX_DELTA):
                    continue

                if MIN_DTE <= dte <= MAX_DTE:
                    resultados.append({
                        "ticker": ticker_raiz,
                        "opcion": details.get("ticker"),
                        "contract_type": details.get("contract_type"),
                        "strike": details.get("strike_price"),
                        "vto": expiry,
                        "dte": dte,

                        # Greeks
                        "delta": round(delta, 4),
                        "gamma": greeks.get("gamma"),
                        "theta": greeks.get("theta"),
                        "vega": greeks.get("vega"),

                        # Volatilidad y liquidez
                        "iv": c.get("implied_volatility"),
                        "oi": c.get("open_interest", 0),
                        "volume": day.get("volume", 0),
                        "vwap": day.get("vwap"),
                        "close_price": day.get("close"),

                        "fecha": date.today()
                    })

            next_url = data.get("next_url")
            url = f"{next_url}&apiKey={POLYGON_API_KEY}" if next_url else None

        except Exception as e:
            print(f"Excepción {ticker_raiz}: {e}")
            break

    return resultados

# ---------------------------
# 3️⃣ Procesar e insertar
# ---------------------------
total_insertados = 0

for ticker in tickers_validos:
    print(f"\nProcesando {ticker}...")
    data = buscar_opciones_por_ticker(ticker)

    if data:
        df = pd.DataFrame(data)
        df.to_sql(
            "salud_opciones",
            engine,
            schema="volatilidad",
            if_exists="append",
            index=False
        )
        print(f"{len(df)} opciones insertadas")
        total_insertados += len(df)
    else:
        print("No se encontraron PUTs válidos")

    time.sleep(0.25)

print(f"\nProceso finalizado. Total opciones insertadas: {total_insertados}")


Procesando 468 tickers multifactor top20...

Procesando LMB...
1 opciones insertadas

Procesando IBEX...
1 opciones insertadas

Procesando LNKB...
No se encontraron PUTs válidos

Procesando SN...
1 opciones insertadas

Procesando EXPD...
No se encontraron PUTs válidos

Procesando ULS...
1 opciones insertadas

Procesando UPBD...
No se encontraron PUTs válidos

Procesando FFIN...
No se encontraron PUTs válidos

Procesando GWW...
3 opciones insertadas

Procesando HBB...
1 opciones insertadas

Procesando CRVL...
No se encontraron PUTs válidos

Procesando BOOT...
2 opciones insertadas

Procesando V...
4 opciones insertadas

Procesando TRV...
1 opciones insertadas

Procesando SSSS...
No se encontraron PUTs válidos

Procesando IPAR...
1 opciones insertadas

Procesando AVGO...
4 opciones insertadas

Procesando PFE...
1 opciones insertadas

Procesando TDW...
No se encontraron PUTs válidos

Procesando CARG...
1 opciones insertadas

Procesando VRTX...
4 opciones insertadas

Procesando TER...
4 op


Procesando CF...
5 opciones insertadas

Procesando CCBG...
No se encontraron PUTs válidos

Procesando YELP...
1 opciones insertadas

Procesando COCO...
1 opciones insertadas

Procesando CHCI...
No se encontraron PUTs válidos

Procesando ALRS...
1 opciones insertadas

Procesando TXRH...
2 opciones insertadas

Procesando ROL...
1 opciones insertadas

Procesando CINF...
1 opciones insertadas

Procesando ECL...
1 opciones insertadas

Procesando BRBR...
No se encontraron PUTs válidos

Procesando MOD...
1 opciones insertadas

Procesando MLI...
No se encontraron PUTs válidos

Procesando EPD...
2 opciones insertadas

Procesando JNJ...
2 opciones insertadas

Procesando PRMB...
1 opciones insertadas

Procesando MET...
No se encontraron PUTs válidos

Procesando PMTS...
No se encontraron PUTs válidos

Procesando GBLI...
No se encontraron PUTs válidos

Procesando MGY...
No se encontraron PUTs válidos

Procesando IRWD...
1 opciones insertadas

Procesando HTB...
No se encontraron PUTs válidos

Proce

No se encontraron PUTs válidos

Procesando EFSC...
No se encontraron PUTs válidos

Procesando PLTR...
4 opciones insertadas

Procesando PRDO...
1 opciones insertadas

Procesando LFVN...
No se encontraron PUTs válidos

Procesando OLED...
1 opciones insertadas

Procesando UBER...
4 opciones insertadas

Procesando PGC...
No se encontraron PUTs válidos

Procesando CRAI...
1 opciones insertadas

Procesando AIT...
1 opciones insertadas

Procesando FELE...
No se encontraron PUTs válidos

Procesando PAYX...
1 opciones insertadas

Procesando TTI...
1 opciones insertadas

Procesando SNV...
2 opciones insertadas

Procesando APA...
3 opciones insertadas

Procesando UVSP...
No se encontraron PUTs válidos

Procesando GLXY...
5 opciones insertadas

Procesando BRC...
No se encontraron PUTs válidos

Procesando EPAC...
No se encontraron PUTs válidos

Procesando MNKD...
1 opciones insertadas

Procesando CASH...
No se encontraron PUTs válidos

Procesando RIGL...
1 opciones insertadas

Procesando ACIW...
N